# LoRa Signal Denoising with Basic MultiBAM

**실험 조건**
1. **모델**: 가장 기본적인 `MultiBAM` (Numpy 버전, Epoch 없음)
2. **데이터 전처리**: Z-score Normalization (평균 0, 표준편차 1)
3. **아키텍처**: $2^i$ 차원으로 깊어지는 층 (Feature Extraction $\ll$ $\gg$ Reconstruction)
4. **학습**: One-pass Learning (데이터를 한 번만 훑음)

In [ ]:
import numpy as np
import glob
import os
import sys
import matplotlib.pyplot as plt

# 상위 폴더 경로 추가 (utils 폴더의 모듈 로드용)
sys.path.append(os.path.abspath(os.path.join(os.getcwd(), '..')))

# ★ 가장 기본 버전의 MultiBAM 로드 (utils/LoRa.py)
from utils.LoRa import MultiBAM

In [ ]:
# 설정
DATASET_FOLDER = "dataset_new_sf9"  # 데이터셋 폴더명 (환경에 맞게 수정)
WEIGHT_FOLDER = "weights_basic_bam"

if not os.path.exists(WEIGHT_FOLDER):
    os.makedirs(WEIGHT_FOLDER)

# 데이터 로드 함수
def load_data(folder_path):
    files = glob.glob(os.path.join(folder_path, '*.npy'))
    if not files:
        # 파일이 없으면 테스트용 랜덤 데이터 생성
        print("⚠️ 데이터 파일이 없어 랜덤 데이터를 생성합니다.")
        return np.random.randn(100, 256*15)
    
    data_list = [np.load(f) for f in files]
    # Flatten (N, 3840)
    X = np.array([d.flatten() for d in data_list])
    return X

# Z-score 정규화 함수
def z_score_normalize(X):
    mean = np.mean(X, axis=0)
    std = np.std(X, axis=0)
    std[std == 0] = 1e-10  # 나눗셈 에러 방지
    return (X - mean) / std, mean, std

# 실행
X_raw = load_data(DATASET_FOLDER)
X_norm, mean_val, std_val = z_score_normalize(X_raw)

# 정규화 값 저장 (나중에 복원 시 사용)
np.save(f"{WEIGHT_FOLDER}/mean.npy", mean_val)
np.save(f"{WEIGHT_FOLDER}/std.npy", std_val)

print(f"Data Loaded: {X_norm.shape}")

In [ ]:
input_dim = X_norm.shape[1]

# 2^i 레이어 구성 (예: 3840 -> 2048 -> 1024 -> 512 -> 256)
# 입력보다 작은 가장 가까운 2의 제곱수부터 시작해 256까지 줄어듦
layers = [input_dim]
current_dim = 2048  # 시작 차원 (2^11)

while current_dim >= 256:
    layers.append(current_dim)
    current_dim //= 2

print("Designed Architecture (Layers):", layers)

In [ ]:
ETA = 1e-3  # 학습률 (발산 시 줄일 것)

# 모델 초기화
multibam = MultiBAM(layers_dims=layers, eta=ETA)

print(f"Start Training with eta={ETA}...")

# 학습 (Epoch 루프 없음, One-pass)
try:
    multibam.train(X_norm)
    print("Training Completed Successfully.")
    
    # 가중치 저장
    for i, bam in enumerate(multibam.bams):
        np.save(f"{WEIGHT_FOLDER}/weights_layer_{i}.npy", bam.W)
        
        # 가중치 상태 확인
        if np.isnan(bam.W).any():
            print(f"⚠️ Layer {i} weights contain NaN! -> Reduce eta.")
        else:
            print(f"Layer {i} saved. Shape: {bam.W.shape}")
            
except Exception as e:
    print(f"Training Error: {e}")

In [ ]:
# 테스트 샘플 (5개)
sample = X_norm[:5]

# 압축 (Feature Extraction)
compressed = multibam.compress(sample)
print(f"Compressed Shape: {compressed.shape}")

# 복원 (Reconstruction)
reconstructed = multibam.decompress(compressed)

# MSE 계산
mse = np.mean((sample - reconstructed) ** 2)
print(f"Reconstruction MSE: {mse:.6f}")

# 시각화 (첫 번째 샘플)
plt.figure(figsize=(12, 5))
plt.plot(sample[0], label='Original (Norm)', alpha=0.7)
plt.plot(reconstructed[0], label='Reconstructed', alpha=0.7, linestyle='--')
plt.title("Signal Reconstruction Result")
plt.legend()
plt.show()